[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/097.%20The%20Last-Mile%20Delivery%20%26%20Micro-Fulfillment%20Problem/P97-Tier-7_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 97. The Last-Mile Delivery & Micro-Fulfillment Problem
## Tier 7: The Human-AI Symbiotic Partnership (The Centaur Model)

### Purpose and scope
This notebook illustrates a human-AI collaboration process for last-mile delivery decisions. The AI component generates a recommendation, reasoning, alternatives, and estimated outcomes. A human operator adds contextual and ethical judgment. A collaboration rule then assigns a partnership role and records a simulated outcome.

The notebook is a conceptual simulation, not a trained or empirically validated AI system. Its scores and decision rules are illustrative heuristics; they should not be interpreted as measured operational performance or evidence that this approach outperforms human-only or AI-only systems. The simulation does not optimize delivery routes, use real customer data, or estimate real-world causal effects.

### Key assumptions
- Human operators can contribute contextual knowledge, empathy, creativity, and ethical judgment that may not be represented in the system inputs.
- AI handles structured analysis and routine recommendation generation.
- Explanations are presented to support human review; they do not prove that a recommendation is correct.
- Trust is represented by a simple score that changes in response to simulated outcomes.
- The human-AI role depends on decision type, confidence, and trust, while a human operator retains the ability to make a different decision.

### Approach
1. Define delivery decisions, operator roles, and collaboration outcomes.
2. Generate an explainable AI recommendation for each decision.
3. Generate a human assessment that incorporates contextual and ethical considerations.
4. Apply decision-type rules to determine the partnership role and final decision.
5. Estimate illustrative outcomes, update trust, and summarize the simulated collaborations.

### Modeling notes
A confidence or performance score is a number on the unit interval $[0,1]$. It is a model output, not a probability or empirical measurement unless it has been calibrated and validated against suitable data. The simulated agreement values are generated independently by simple random perturbations; they are not computed by comparing the text of the AI and human decisions.

The trust update is a threshold-based rule. For operator trust $T_t$ after collaboration $t$, customer satisfaction $S_t$, and operational efficiency $E_t$:

$$
T_{t+1} =
\begin{cases}
\min(1, T_t + 0.02), & S_t > 0.85 \text{ and } E_t > 0.80, \\
\max(0, T_t - 0.05), & S_t < 0.70 \text{ or } E_t < 0.70, \\
T_t, & \text{otherwise.}
\end{cases}
$$

Here, $T_t$ is implemented as `human_operator.trust_score`, and $S_t$ and $E_t$ are the outcome fields `customer_satisfaction` and `operational_efficiency`. The strict inequalities are part of the rule. This update is a teaching abstraction, not a validated model of how trust develops in real teams.

### Concrete example reported in the problem description
The problem description gives an illustrative holiday-season case involving Sarah Martinez, 23 exceptional situations, 156,000 AI calculations, and 96.8% customer satisfaction. These figures are reported scenario details, not results generated or verified by the simulation below. The notebook's small scenario set cannot reproduce or validate the reported holiday-season totals.

### Interpreting the results
The results describe the behavior of the specified rules under simulated inputs. Random variation means that outcome scores and operator assignments can vary between runs. The summaries are useful for examining how the model routes decision types and updates trust, but they are not a statistical comparison of human-AI collaboration with other operating models.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from enum import Enum
from typing import Dict, List, Tuple
import random

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
class DecisionType(Enum):
    ROUTINE = 'routine'
    CONTEXTUAL = 'contextual'
    EXCEPTIONAL = 'exceptional'
    ETHICAL = 'ethical'


class PartnershipRole(Enum):
    HUMAN_LEAD = 'human_lead'
    AI_LEAD = 'ai_lead'
    COLLABORATIVE = 'collaborative'


@dataclass
class DeliveryDecision:
    """A delivery decision considered by the human-AI partnership."""

    id: int
    decision_type: DecisionType
    description: str
    ai_recommendation: Dict
    human_context: Dict
    complexity_score: float
    time_sensitivity: float
    customer_impact: float


@dataclass
class AIDecision:
    """AI recommendation with an explanation and illustrative estimates."""

    recommendation: str
    confidence: float
    reasoning: List[str]
    alternatives: List[Dict]
    estimated_outcomes: Dict[str, float]
    ethical_considerations: List[str]


@dataclass
class HumanDecision:
    """Human assessment including contextual and ethical considerations."""

    decision: str
    reasoning: str
    contextual_factors: List[str]
    ethical_concerns: List[str]
    experience_based_insights: str
    confidence: float


@dataclass
class CollaborationOutcome:
    """Recorded outcome of one simulated collaboration."""

    final_decision: str
    partnership_role: PartnershipRole
    human_agreement: float
    ai_agreement: float
    decision_time: timedelta
    customer_satisfaction: float
    operational_efficiency: float


class ExplainableAI:
    """Heuristic recommendation generator used in this teaching simulation."""

    def __init__(self):
        self.decision_history = []
        self.performance_metrics = {
            'accuracy': 0.85,
            'reliability': 0.90,
            'transparency': 0.95,
        }

    def analyze_decision(self, decision: DeliveryDecision) -> AIDecision:
        """Create a recommendation and supporting illustrative information."""
        base_confidence = 0.8
        confidence_adjustments = {
            DecisionType.ROUTINE: 0.15,
            DecisionType.CONTEXTUAL: -0.05,
            DecisionType.EXCEPTIONAL: -0.15,
            DecisionType.ETHICAL: -0.10,
        }
        base_confidence += confidence_adjustments[decision.decision_type]
        confidence = max(
            0.3,
            min(0.95, base_confidence + random.uniform(-0.1, 0.1)),
        )

        return AIDecision(
            recommendation=self._make_recommendation(decision),
            confidence=confidence,
            reasoning=self._generate_reasoning(decision),
            alternatives=self._generate_alternatives(decision),
            estimated_outcomes=self._estimate_outcomes(decision),
            ethical_considerations=self._identify_ethical_considerations(decision),
        )

    def _generate_reasoning(self, decision: DeliveryDecision) -> List[str]:
        """Return illustrative reasoning statements for the decision category."""
        reasoning_by_type = {
            DecisionType.ROUTINE: [
                'Historical data shows this pattern occurs 85% of the time.',
                'A standard operating procedure applies to this situation.',
                'The illustrative optimization estimate gives a 92% success probability.',
                'The modeled cost-benefit comparison favors the recommended approach.',
            ],
            DecisionType.CONTEXTUAL: [
                'Traffic patterns indicate potential delays in the area.',
                'Customer history may indicate a preference for faster delivery.',
                'Vehicle availability is limited during this time window.',
                'An alternative route may increase cost while improving reliability.',
            ],
            DecisionType.EXCEPTIONAL: [
                'This situation falls outside normal operating parameters.',
                'Time, cost, and service-quality constraints may conflict.',
                'The illustrative risk assessment indicates a moderate service-failure risk.',
                'A creative problem-solving approach may be required.',
            ],
            DecisionType.ETHICAL: [
                'The decision involves trade-offs among stakeholder interests.',
                'Fairness must be considered alongside efficiency.',
                'Long-term relationship effects may differ from short-term operating needs.',
                'Company values and ethical guidelines provide a decision framework.',
            ],
        }
        return reasoning_by_type[decision.decision_type]

    def _generate_alternatives(self, decision: DeliveryDecision) -> List[Dict]:
        """Return example alternatives; their values are illustrative, not observed."""
        if decision.decision_type == DecisionType.ROUTINE:
            return [
                {'option': 'Standard route', 'cost': 1.0, 'time': 1.0, 'reliability': 0.95},
                {'option': 'Express route', 'cost': 1.3, 'time': 0.7, 'reliability': 0.90},
                {'option': 'Economy route', 'cost': 0.8, 'time': 1.4, 'reliability': 0.85},
            ]
        if decision.decision_type == DecisionType.CONTEXTUAL:
            return [
                {'option': 'Prioritize speed', 'cost': 1.4, 'time': 0.6, 'customer_satisfaction': 0.90},
                {'option': 'Balance factors', 'cost': 1.0, 'time': 1.0, 'customer_satisfaction': 0.85},
                {'option': 'Prioritize cost', 'cost': 0.7, 'time': 1.5, 'customer_satisfaction': 0.70},
            ]
        return [
            {'option': 'Conservative approach', 'risk': 0.2, 'reward': 0.6, 'stakeholder_impact': 'low'},
            {'option': 'Moderate approach', 'risk': 0.5, 'reward': 0.8, 'stakeholder_impact': 'medium'},
            {'option': 'Aggressive approach', 'risk': 0.8, 'reward': 0.9, 'stakeholder_impact': 'high'},
        ]

    def _estimate_outcomes(self, decision: DeliveryDecision) -> Dict[str, float]:
        """Generate bounded illustrative outcome scores."""
        outcomes = {
            'success_probability': 0.85,
            'customer_satisfaction': 0.80,
            'operational_efficiency': 0.85,
            'cost_effectiveness': 0.80,
        }
        complexity_adjustment = 1.0 - (decision.complexity_score * 0.2)
        for key, value in outcomes.items():
            outcomes[key] = max(
                0.3,
                min(0.95, value * complexity_adjustment + random.uniform(-0.05, 0.05)),
            )
        return outcomes

    def _identify_ethical_considerations(self, decision: DeliveryDecision) -> List[str]:
        """List ethical considerations associated with a decision category."""
        if decision.decision_type in (DecisionType.ETHICAL, DecisionType.EXCEPTIONAL):
            return [
                'Impact on customer trust and the relationship.',
                'Fairness to all stakeholders involved.',
                'Alignment with company values and mission.',
                'Long-term consequences compared with short-term gains.',
            ]
        if decision.decision_type == DecisionType.CONTEXTUAL:
            return [
                'Consistency in the treatment of similar situations.',
                'Fairness in resource allocation.',
                'Transparency in communication with the customer.',
            ]
        return [
            'Compliance with standard operating procedures.',
            'Quality-assurance requirements.',
        ]

    def _make_recommendation(self, decision: DeliveryDecision) -> str:
        """Select an illustrative recommendation for the decision category."""
        recommendations = {
            DecisionType.ROUTINE: [
                'Proceed with the standard delivery route and timing.',
                'Assign a vehicle using the availability procedure.',
                'Apply normal quality-assurance procedures.',
            ],
            DecisionType.CONTEXTUAL: [
                'Prioritize customer satisfaction with an express delivery option.',
                'Allocate additional resources to support reliability.',
                'Communicate proactively with the customer about delivery status.',
            ],
            DecisionType.EXCEPTIONAL: [
                'Combine delivery methods to develop a creative solution.',
                'Escalate special handling for senior review.',
                'Document the decision for future reference and learning.',
            ],
            DecisionType.ETHICAL: [
                'Choose an option that balances stakeholder interests fairly.',
                'Prioritize the long-term relationship over short-term efficiency.',
                'Ensure the decision aligns with company ethical guidelines.',
            ],
        }
        return random.choice(recommendations[decision.decision_type])


class HumanOperator:
    """Human operator represented by experience, specialization, and trust."""

    def __init__(self, name: str, experience_years: int, specialization: str):
        self.name = name
        self.experience_years = experience_years
        self.specialization = specialization
        self.trust_score = 0.5
        self.decision_history = []
        self.expertise_areas = [
            'customer relationship management',
            'cultural sensitivity',
            'creative problem-solving',
            'ethical judgment',
            'local knowledge',
        ]

    def analyze_decision(
        self, decision: DeliveryDecision, ai_decision: AIDecision
    ) -> HumanDecision:
        """Assess the decision using the operator's simulated experience."""
        base_confidence = 0.7 + (self.experience_years / 20) * 0.2
        confidence_adjustments = {
            DecisionType.ROUTINE: -0.1,
            DecisionType.CONTEXTUAL: 0.1,
            DecisionType.EXCEPTIONAL: 0.15,
            DecisionType.ETHICAL: 0.2,
        }
        base_confidence += confidence_adjustments[decision.decision_type]
        confidence = max(
            0.4,
            min(0.95, base_confidence + random.uniform(-0.1, 0.1)),
        )
        return HumanDecision(
            decision=self._make_human_decision(decision, ai_decision),
            reasoning=self._generate_human_reasoning(decision),
            contextual_factors=self._identify_contextual_factors(decision),
            ethical_concerns=self._identify_human_ethical_concerns(decision),
            experience_based_insights=self._generate_experience_insights(decision),
            confidence=confidence,
        )

    def _generate_human_reasoning(self, decision: DeliveryDecision) -> str:
        """Describe a human perspective for the decision category."""
        reasoning = {
            DecisionType.ROUTINE: (
                'The AI recommendation appears suitable for this standard situation. '
                'Experience suggests that established procedures can work well here.'
            ),
            DecisionType.CONTEXTUAL: (
                'The AI has useful data, but local context may be missing. '
                'Customer history and conditions in the area should also be considered.'
            ),
            DecisionType.EXCEPTIONAL: (
                'This situation calls for human judgment. The AI may have incomplete '
                "information about the customer's needs and local circumstances."
            ),
            DecisionType.ETHICAL: (
                'The analysis should account for effects on customers and the community, '
                'including long-term consequences beyond immediate operating metrics.'
            ),
        }
        return reasoning[decision.decision_type]

    def _identify_contextual_factors(self, decision: DeliveryDecision) -> List[str]:
        """List contextual factors that may matter to a human review."""
        factors = {
            DecisionType.ROUTINE: [
                'Standard operating conditions apply.',
                'No special contextual factors are identified in this scenario.',
            ],
            DecisionType.CONTEXTUAL: [
                'Local road construction may not appear in the traffic system.',
                "The customer's business hours and site accessibility may matter.",
                'Weather can vary across a local area.',
                'Community events may affect delivery routes.',
            ],
            DecisionType.EXCEPTIONAL: [
                "The customer's emergency circumstances may affect the response.",
                'Building access may impose additional requirements.',
                'Cultural considerations may be relevant.',
                'Previous service issues may affect the relationship.',
            ],
            DecisionType.ETHICAL: [
                'Community impact and perception may matter.',
                'Long-term customer relationships should be considered.',
                'Company reputation in the area may be affected.',
                'The wider market context may influence the decision.',
            ],
        }
        return factors[decision.decision_type]

    def _identify_human_ethical_concerns(self, decision: DeliveryDecision) -> List[str]:
        """List ethical concerns from the human review perspective."""
        if decision.decision_type == DecisionType.ETHICAL:
            return [
                'Fair treatment of customers regardless of their commercial value.',
                'Effects on employee workload and safety.',
                'Environmental responsibility and sustainability.',
                'Community relationships and social license to operate.',
            ]
        if decision.decision_type == DecisionType.EXCEPTIONAL:
            return [
                'Precedent for future similar situations.',
                'Consistency in policy application.',
                'Fairness in resource allocation.',
            ]
        return [
            'Standard ethical practices apply.',
            'No additional major ethical concern is specified in this scenario.',
        ]

    def _generate_experience_insights(self, decision: DeliveryDecision) -> str:
        """Generate a qualitative, illustrative experience statement."""
        similar_cases = random.randint(20, 100)
        if decision.decision_type == DecisionType.ROUTINE:
            insight = 'Efficiency and consistency are important.'
            action = 'Established patterns may be suitable.'
        elif decision.decision_type == DecisionType.EXCEPTIONAL:
            insight = 'Human relationships and context are important.'
            action = 'The response may need to go beyond standard procedures.'
        else:
            insight = 'Human relationships and context are important.'
            action = 'Established procedures should be considered alongside the specific circumstances.'
        return (
            f'Based on {self.experience_years} years of experience, '
            f'I have seen {similar_cases} similar situations. {insight} {action}'
        )

    def _make_human_decision(
        self, decision: DeliveryDecision, ai_decision: AIDecision
    ) -> str:
        """Choose a human response based on decision type and current trust."""
        if decision.decision_type == DecisionType.ROUTINE:
            if self.trust_score > 0.7:
                return ai_decision.recommendation
            return random.choice([
                ai_decision.recommendation,
                'Follow the standard procedure with minor adjustments.',
            ])
        if decision.decision_type == DecisionType.CONTEXTUAL:
            if self.trust_score > 0.6:
                return f'AI recommendation with human adjustment: {ai_decision.recommendation}'
            return 'Prioritize human contextual understanding over the AI recommendation.'
        if decision.decision_type == DecisionType.EXCEPTIONAL:
            return 'Use a human-led creative solution informed by experience and context.'
        return 'Apply human ethical judgment with analytical support from the AI.'


class HumanAIPartnership:
    """Coordinate simulated AI recommendations and human review."""

    def __init__(self):
        self.ai_system = ExplainableAI()
        self.human_operators = []
        self.partnership_history = []
        self.trust_metrics = []
        self.performance_metrics = {
            'collaboration_effectiveness': 0.0,
            'decision_quality': 0.0,
            'customer_satisfaction': 0.0,
            'operational_efficiency': 0.0,
        }

    def add_human_operator(
        self, name: str, experience_years: int, specialization: str
    ) -> None:
        """Register an operator who can be selected for a decision."""
        operator = HumanOperator(name, experience_years, specialization)
        self.human_operators.append(operator)
        print(
            f'Added human operator: {name} ({experience_years} years experience, '
            f'{specialization})'
        )

    def collaborate_on_decision(
        self, decision: DeliveryDecision
    ) -> CollaborationOutcome:
        """Run the AI analysis, human review, role rule, and trust update."""
        start_time = datetime.now()
        ai_decision = self.ai_system.analyze_decision(decision)
        human_operator = self._select_human_operator(decision)
        human_decision = human_operator.analyze_decision(decision, ai_decision)
        partnership_role, final_decision = self._determine_partnership_role(
            decision, ai_decision, human_decision, human_operator
        )
        human_agreement = self._calculate_agreement(
            ai_decision, human_decision, 'human'
        )
        ai_agreement = self._calculate_agreement(
            ai_decision, human_decision, 'ai'
        )
        customer_satisfaction = self._estimate_customer_satisfaction(
            decision, partnership_role
        )
        operational_efficiency = self._estimate_operational_efficiency(
            decision, partnership_role
        )
        outcome = CollaborationOutcome(
            final_decision=final_decision,
            partnership_role=partnership_role,
            human_agreement=human_agreement,
            ai_agreement=ai_agreement,
            decision_time=datetime.now() - start_time,
            customer_satisfaction=customer_satisfaction,
            operational_efficiency=operational_efficiency,
        )

        self._update_trust_and_learning(human_operator, outcome, decision)
        self.partnership_history.append({
            'decision_id': decision.id,
            'decision_type': decision.decision_type.value,
            'partnership_role': partnership_role.value,
            'human_operator': human_operator.name,
            'outcome': outcome,
            'timestamp': datetime.now(),
        })
        return outcome

    def _select_human_operator(self, decision: DeliveryDecision) -> HumanOperator:
        """Select an operator, preferring experience for ethical decisions."""
        if not self.human_operators:
            return HumanOperator('Default Operator', 5, 'General')
        if decision.decision_type == DecisionType.ETHICAL:
            experienced = [
                operator for operator in self.human_operators
                if operator.experience_years > 10
            ]
            if experienced:
                return random.choice(experienced)
        return random.choice(self.human_operators)

    def _determine_partnership_role(
        self,
        decision: DeliveryDecision,
        ai_decision: AIDecision,
        human_decision: HumanDecision,
        human_operator: HumanOperator,
    ) -> Tuple[PartnershipRole, str]:
        """Apply the role rules for the decision category."""
        if decision.decision_type == DecisionType.ROUTINE:
            if human_operator.trust_score > 0.7 and human_decision.confidence > 0.8:
                return PartnershipRole.AI_LEAD, ai_decision.recommendation
            return (
                PartnershipRole.COLLABORATIVE,
                f'AI recommendation with human oversight: {ai_decision.recommendation}',
            )
        if decision.decision_type == DecisionType.CONTEXTUAL:
            if human_decision.confidence > ai_decision.confidence:
                return PartnershipRole.COLLABORATIVE, human_decision.decision
            return PartnershipRole.COLLABORATIVE, 'Balanced human-AI decision.'
        if decision.decision_type in (
            DecisionType.EXCEPTIONAL, DecisionType.ETHICAL
        ):
            return PartnershipRole.HUMAN_LEAD, human_decision.decision
        raise ValueError(f'Unsupported decision type: {decision.decision_type}')

    def _calculate_agreement(
        self, ai_decision: AIDecision, human_decision: HumanDecision,
        perspective: str,
    ) -> float:
        """Generate a bounded illustrative agreement score."""
        if perspective == 'human':
            agreement = 0.7 + random.uniform(-0.2, 0.2)
        elif perspective == 'ai':
            agreement = 0.7 + random.uniform(-0.15, 0.15)
        else:
            raise ValueError("perspective must be 'human' or 'ai'")
        return max(0.0, min(1.0, agreement))

    def _estimate_customer_satisfaction(
        self, decision: DeliveryDecision, partnership_role: PartnershipRole
    ) -> float:
        """Calculate a bounded illustrative satisfaction score."""
        satisfaction = 0.80
        if partnership_role == PartnershipRole.HUMAN_LEAD:
            satisfaction += 0.10
        elif partnership_role == PartnershipRole.COLLABORATIVE:
            satisfaction += 0.05
        if decision.decision_type == DecisionType.EXCEPTIONAL:
            satisfaction += 0.10
        elif decision.decision_type == DecisionType.ETHICAL:
            satisfaction += 0.05
        return max(0.0, min(1.0, satisfaction + random.uniform(-0.05, 0.05)))

    def _estimate_operational_efficiency(
        self, decision: DeliveryDecision, partnership_role: PartnershipRole
    ) -> float:
        """Calculate a bounded illustrative efficiency score."""
        efficiency = 0.85
        if partnership_role == PartnershipRole.AI_LEAD:
            efficiency += 0.10
        elif partnership_role == PartnershipRole.COLLABORATIVE:
            efficiency += 0.05
        if decision.decision_type == DecisionType.ROUTINE:
            efficiency += 0.05
        elif decision.decision_type == DecisionType.EXCEPTIONAL:
            efficiency -= 0.10
        return max(0.0, min(1.0, efficiency + random.uniform(-0.05, 0.05)))

    def _update_trust_and_learning(
        self,
        human_operator: HumanOperator,
        outcome: CollaborationOutcome,
        decision: DeliveryDecision,
    ) -> None:
        """Update bounded trust and retain a record for the trust analysis."""
        if (
            outcome.customer_satisfaction > 0.85
            and outcome.operational_efficiency > 0.80
        ):
            human_operator.trust_score = min(
                1.0, human_operator.trust_score + 0.02
            )
        elif (
            outcome.customer_satisfaction < 0.70
            or outcome.operational_efficiency < 0.70
        ):
            human_operator.trust_score = max(
                0.0, human_operator.trust_score - 0.05
            )
        self.trust_metrics.append({
            'operator': human_operator.name,
            'trust_score': human_operator.trust_score,
            'decision_type': decision.decision_type.value,
            'partnership_role': outcome.partnership_role.value,
            'timestamp': datetime.now(),
        })


print('Human-AI Symbiotic Partnership implementation complete.')

In [ ]:
# Create the partnership and register the operators used in the scenarios.
print('=' * 60)
print('INITIALIZING HUMAN-AI SYMBIOTIC PARTNERSHIP')
print('=' * 60)

partnership = HumanAIPartnership()
partnership.add_human_operator('Sarah Martinez', 12, 'Customer Relations')
partnership.add_human_operator('John Chen', 8, 'Operations')
partnership.add_human_operator('Maria Garcia', 15, 'Ethics & Compliance')
partnership.add_human_operator('David Kim', 6, 'Logistics')
partnership.add_human_operator('Emily Johnson', 10, 'Problem Solving')

expertise_areas = {
    area
    for operator in partnership.human_operators
    for area in operator.expertise_areas
}
print('\nPARTNERSHIP STATUS:')
print(f'Human operators: {len(partnership.human_operators)}')
print('AI system: Explainable AI (heuristic simulation)')
print(f'Total listed expertise areas: {len(expertise_areas)}')

print('\nHUMAN OPERATORS:')
for operator in partnership.human_operators:
    print(
        f'{operator.name}: {operator.experience_years} years of experience, '
        f'{operator.specialization}; initial trust score: '
        f'{operator.trust_score:.2f}'
    )

In [ ]:
# Define a small set of scenarios spanning the four decision categories.
print('=' * 60)
print('GENERATING DELIVERY DECISION SCENARIOS')
print('=' * 60)

decisions = [
    DeliveryDecision(
        id=1, decision_type=DecisionType.ROUTINE,
        description='Standard delivery route optimization',
        ai_recommendation={'route': 'standard', 'vehicle': 'van'},
        human_context={'customer': 'regular', 'area': 'normal'},
        complexity_score=0.2, time_sensitivity=0.3, customer_impact=0.4,
    ),
    DeliveryDecision(
        id=2, decision_type=DecisionType.ROUTINE,
        description='Vehicle assignment for multiple deliveries',
        ai_recommendation={'assignment': 'algorithm', 'efficiency': 'high'},
        human_context={'workload': 'normal', 'resources': 'available'},
        complexity_score=0.3, time_sensitivity=0.4, customer_impact=0.3,
    ),
    DeliveryDecision(
        id=3, decision_type=DecisionType.CONTEXTUAL,
        description='Delivery through downtown during a festival',
        ai_recommendation={'route': 'highway', 'time': '45 min'},
        human_context={'cultural_event': 'festival', 'community_impact': 'high'},
        complexity_score=0.6, time_sensitivity=0.7, customer_impact=0.8,
    ),
    DeliveryDecision(
        id=4, decision_type=DecisionType.CONTEXTUAL,
        description='Weather disruption routing decision',
        ai_recommendation={'reroute': 'yes', 'delay': '20 min'},
        human_context={'weather': 'storm', 'safety': 'concern'},
        complexity_score=0.5, time_sensitivity=0.8, customer_impact=0.6,
    ),
    DeliveryDecision(
        id=5, decision_type=DecisionType.EXCEPTIONAL,
        description='Building access emergency situation',
        ai_recommendation={'escalate': 'yes', 'alternative': 'pickup point'},
        human_context={'emergency': 'medical', 'access': 'restricted'},
        complexity_score=0.9, time_sensitivity=0.9, customer_impact=0.9,
    ),
    DeliveryDecision(
        id=6, decision_type=DecisionType.EXCEPTIONAL,
        description='Multiple failed delivery attempts',
        ai_recommendation={'retry': 'no', 'refund': 'process'},
        human_context={'customer': 'vip', 'relationship': 'long-term'},
        complexity_score=0.8, time_sensitivity=0.7, customer_impact=0.8,
    ),
    DeliveryDecision(
        id=7, decision_type=DecisionType.ETHICAL,
        description='Resource allocation during a shortage',
        ai_recommendation={'prioritize': 'high_value', 'efficiency': 'max'},
        human_context={'fairness': 'concern', 'community': 'impact'},
        complexity_score=0.7, time_sensitivity=0.6, customer_impact=0.8,
    ),
    DeliveryDecision(
        id=8, decision_type=DecisionType.ETHICAL,
        description='Customer privacy versus delivery efficiency',
        ai_recommendation={'optimize': 'route', 'data': 'use'},
        human_context={'privacy': 'rights', 'trust': 'essential'},
        complexity_score=0.6, time_sensitivity=0.5, customer_impact=0.7,
    ),
]

decision_counts = {}
for decision in decisions:
    category = decision.decision_type.value
    decision_counts[category] = decision_counts.get(category, 0) + 1

print(f'Scenarios defined: {len(decisions)}')
for decision_type, count in decision_counts.items():
    print(f'{decision_type.title()}: {count} decisions')

# This separate scenario represents the holiday example described above.
holiday_decision = DeliveryDecision(
    id=100,
    decision_type=DecisionType.CONTEXTUAL,
    description='Holiday delivery routing with cultural considerations',
    ai_recommendation={'route': 'downtown_optimal', 'time': '17 min'},
    human_context={
        'holiday': 'Christmas',
        'cultural_district': 'festival',
        'relations': 'community',
    },
    complexity_score=0.7,
    time_sensitivity=0.8,
    customer_impact=0.9,
)
print('\nHOLIDAY-SEASON EXAMPLE:')
print(f'Decision: {holiday_decision.description}')
print('AI recommendation input: downtown route, 17 minutes')
print('Human context input: festival in a cultural district; community relations matter')
print(
    f'Complexity: {holiday_decision.complexity_score:.1f}; '
    f'time sensitivity: {holiday_decision.time_sensitivity:.1f}'
)

In [ ]:
# Run each defined scenario through the same collaboration process.
print('=' * 60)
print('RUNNING HUMAN-AI COLLABORATION')
print('=' * 60)

collaboration_results = []
for decision in decisions:
    print(f'\n--- Decision {decision.id}: {decision.description} ---')
    print(f'Type: {decision.decision_type.value.title()}')
    outcome = partnership.collaborate_on_decision(decision)
    collaboration_results.append(outcome)
    print(f'Partnership role: {outcome.partnership_role.value.replace("_", " ").title()}')
    print(f'Final decision: {outcome.final_decision}')
    print(f'Human agreement score: {outcome.human_agreement:.1%}')
    print(f'AI agreement score: {outcome.ai_agreement:.1%}')
    print(f'Processing time: {outcome.decision_time.total_seconds():.6f} seconds')
    print(f'Illustrative customer satisfaction score: {outcome.customer_satisfaction:.1%}')
    print(f'Illustrative operational efficiency score: {outcome.operational_efficiency:.1%}')

# The operator is selected by the partnership's selection rule; the name
# shown here is the operator actually assigned to this simulated case.
print('\n' + '=' * 60)
print('SPECIAL HOLIDAY SCENARIO')
print('=' * 60)
holiday_outcome = partnership.collaborate_on_decision(holiday_decision)
holiday_record = partnership.partnership_history[-1]
print(f'Operator selected: {holiday_record["human_operator"]}')
print('AI recommendation input: downtown route, 17 minutes')
print('Human context input: Christmas festival and cultural-district sensitivity')
print(f'Partnership role: {holiday_outcome.partnership_role.value.replace("_", " ").title()}')
print(f'Final decision: {holiday_outcome.final_decision}')
print(f'Illustrative customer satisfaction score: {holiday_outcome.customer_satisfaction:.1%}')
print(f'Illustrative operational efficiency score: {holiday_outcome.operational_efficiency:.1%}')

print('\nHOLIDAY-SEASON REPORTED FIGURES')
print('The problem description reports 156,000 AI calculations and 23 exceptional situations.')
print('Those figures are scenario details and are not generated by this small simulation.')


In [ ]:
# Summarize the simulated outcomes; these scores are not empirical estimates.
print('=' * 60)
print('COLLABORATION ANALYSIS AND TRUST DEVELOPMENT')
print('=' * 60)

if not collaboration_results:
    raise ValueError('At least one collaboration result is required for analysis.')

avg_customer_satisfaction = np.mean(
    [result.customer_satisfaction for result in collaboration_results]
)
avg_operational_efficiency = np.mean(
    [result.operational_efficiency for result in collaboration_results]
)
avg_human_agreement = np.mean(
    [result.human_agreement for result in collaboration_results]
)
avg_ai_agreement = np.mean(
    [result.ai_agreement for result in collaboration_results]
)
avg_decision_time = np.mean(
    [result.decision_time.total_seconds() for result in collaboration_results]
)

print('\nOVERALL SIMULATION METRICS:')
print(f'Average customer satisfaction score: {avg_customer_satisfaction:.1%}')
print(f'Average operational efficiency score: {avg_operational_efficiency:.1%}')
print(f'Average human agreement score: {avg_human_agreement:.1%}')
print(f'Average AI agreement score: {avg_ai_agreement:.1%}')
print(f'Average processing time: {avg_decision_time:.6f} seconds')

role_counts = {}
role_performance = {}
for outcome in collaboration_results:
    role = outcome.partnership_role.value
    role_counts[role] = role_counts.get(role, 0) + 1
    role_performance.setdefault(role, {'satisfaction': [], 'efficiency': []})
    role_performance[role]['satisfaction'].append(outcome.customer_satisfaction)
    role_performance[role]['efficiency'].append(outcome.operational_efficiency)

print('\nPARTNERSHIP ROLE SUMMARY:')
for role, count in role_counts.items():
    avg_satisfaction = np.mean(role_performance[role]['satisfaction'])
    avg_efficiency = np.mean(role_performance[role]['efficiency'])
    print(f'{role.replace("_", " ").title()}: {count} decisions')
    print(
        f'  Mean illustrative satisfaction: {avg_satisfaction:.1%}; '
        f'mean illustrative efficiency: {avg_efficiency:.1%}'
    )

# Associate each outcome with the decision that produced it.
type_analysis = {}
for decision, outcome in zip(decisions, collaboration_results):
    decision_type = decision.decision_type.value
    type_analysis.setdefault(
        decision_type,
        {'satisfaction': [], 'efficiency': [], 'agreement': []},
    )
    type_analysis[decision_type]['satisfaction'].append(
        outcome.customer_satisfaction
    )
    type_analysis[decision_type]['efficiency'].append(
        outcome.operational_efficiency
    )
    type_analysis[decision_type]['agreement'].append(
        (outcome.human_agreement + outcome.ai_agreement) / 2
    )

print('\nSIMULATED OUTCOMES BY DECISION TYPE:')
for decision_type, metrics in type_analysis.items():
    print(f'{decision_type.title()}:')
    print(
        f'  Satisfaction: {np.mean(metrics["satisfaction"]):.1%}; '
        f'efficiency: {np.mean(metrics["efficiency"]):.1%}; '
        f'mean agreement score: {np.mean(metrics["agreement"]):.1%}'
    )

print('\nTRUST DEVELOPMENT:')
for operator in partnership.human_operators:
    operator_records = [
        record for record in partnership.trust_metrics
        if record['operator'] == operator.name
    ]
    final_trust = operator.trust_score
    trust_change = final_trust - 0.5
    print(f'{operator.name}:')
    print('  Initial trust score: 0.50')
    print(f'  Final trust score: {final_trust:.2f}')
    print(f'  Change from initial score: {trust_change:+.2f}')
    print(f'  Recorded collaborations: {len(operator_records)}')

print('\nLEARNING AND ADAPTATION RECORD:')
print(f'Collaborations recorded: {len(partnership.partnership_history)}')
print(f'Trust updates recorded: {len(partnership.trust_metrics)}')
print(
    'Trust updates follow the stated threshold rule; they do not represent '
    'a learned predictive model.'
)

print('\nREFERENCE TO THE REPORTED PROBLEM-DESCRIPTION CASE:')
print('Reported customer satisfaction: 96.8% (scenario figure, not a simulation result)')
print('Reported AI calculations: 156,000 (scenario figure, not simulated here)')
print('Reported exceptional situations: 23 (scenario figure, not simulated here)')

assessment_score = 0
if avg_customer_satisfaction > 0.90:
    assessment_score += 1
if avg_operational_efficiency > 0.85:
    assessment_score += 1
if avg_human_agreement > 0.70:
    assessment_score += 1
if any(operator.trust_score > 0.8 for operator in partnership.human_operators):
    assessment_score += 1
if len(role_counts) >= 3:
    assessment_score += 1

final_score = assessment_score / 5 * 100
print('\nRULE-BASED SIMULATION CHECKLIST:')
print(f'Criteria met: {assessment_score} of 5')
print(f'Checklist score: {final_score:.0f}%')
print(
    'This checklist summarizes preset thresholds and is not a validated '
    'measure of partnership quality.'
)

In [ ]:
# Visualize role counts, trust records, and simulated performance summaries.
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    'Human-AI Symbiotic Partnership: Simulated Outcomes',
    fontsize=16,
    fontweight='bold',
)

# Partnership role distribution.
ax1 = axes[0, 0]
role_names = list(role_counts.keys())
role_values = [role_counts[role] for role in role_names]
role_color_map = {
    PartnershipRole.AI_LEAD.value: 'lightblue',
    PartnershipRole.COLLABORATIVE.value: 'lightgreen',
    PartnershipRole.HUMAN_LEAD.value: 'lightcoral',
}
role_colors = [role_color_map[role] for role in role_names]
bars = ax1.bar(role_names, role_values, color=role_colors, alpha=0.8, edgecolor='black')
ax1.set_xlabel('Partnership role')
ax1.set_ylabel('Number of decisions')
ax1.set_title('Partnership Role Distribution')
ax1.tick_params(axis='x', rotation=20)
ax1.grid(True, axis='y', alpha=0.3)
for bar, value in zip(bars, role_values):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        str(value),
        ha='center',
        va='bottom',
    )

# Trust records are grouped by operator; x is the recorded collaboration index.
ax2 = axes[0, 1]
for operator in partnership.human_operators:
    operator_trust = [
        record['trust_score']
        for record in partnership.trust_metrics
        if record['operator'] == operator.name
    ]
    if operator_trust:
        ax2.plot(
            range(1, len(operator_trust) + 1),
            operator_trust,
            marker='o',
            linewidth=2,
            label=operator.name,
        )
ax2.set_xlabel('Recorded collaboration number for operator')
ax2.set_ylabel('Trust score')
ax2.set_title('Trust Scores Recorded Over Time')
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)
if partnership.trust_metrics:
    ax2.legend(fontsize=8)

# Mean illustrative scores by decision category.
ax3 = axes[1, 0]
decision_types = list(type_analysis.keys())
satisfaction_by_type = [
    np.mean(type_analysis[category]['satisfaction'])
    for category in decision_types
]
efficiency_by_type = [
    np.mean(type_analysis[category]['efficiency'])
    for category in decision_types
]
x = np.arange(len(decision_types))
width = 0.35
ax3.bar(
    x - width / 2,
    satisfaction_by_type,
    width,
    label='Illustrative customer satisfaction',
    color='skyblue',
    alpha=0.8,
)
ax3.bar(
    x + width / 2,
    efficiency_by_type,
    width,
    label='Illustrative operational efficiency',
    color='lightgreen',
    alpha=0.8,
)
ax3.set_xlabel('Decision type')
ax3.set_ylabel('Score (0 to 1)')
ax3.set_title('Simulated Scores by Decision Type')
ax3.set_xticks(x)
ax3.set_xticklabels([category.title() for category in decision_types], rotation=30)
ax3.set_ylim(0, 1)
ax3.legend(fontsize=8)
ax3.grid(True, axis='y', alpha=0.3)

# The agreement score is defined by the simulation's random-score rule,
# not by textual comparison of the two decisions.
ax4 = axes[1, 1]
average_agreements = [
    np.mean(type_analysis[category]['agreement'])
    for category in decision_types
]
agreement_colors = [
    'red' if score < 0.6 else 'orange' if score < 0.8 else 'green'
    for score in average_agreements
]
bars = ax4.bar(
    np.arange(len(decision_types)),
    average_agreements,
    color=agreement_colors,
    alpha=0.8,
)
ax4.set_xlabel('Decision type')
ax4.set_ylabel('Mean simulated agreement score (0 to 1)')
ax4.set_title('Simulated Agreement Scores by Decision Type')
ax4.set_xticks(np.arange(len(decision_types)))
ax4.set_xticklabels([category.title() for category in decision_types], rotation=30)
ax4.set_ylim(0, 1)
ax4.grid(True, axis='y', alpha=0.3)
ax4.axhline(
    y=0.7,
    color='black',
    linestyle='--',
    alpha=0.6,
    label='Illustrative reference level: 0.70',
)
ax4.legend(fontsize=8)
for bar, score in zip(bars, average_agreements):
    ax4.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f'{score:.2f}',
        ha='center',
        va='bottom',
    )

fig.tight_layout()
plt.show()
print('Simulation visualizations complete.')

## Why this tier exists
The Human-AI Symbiotic Partnership focuses on decisions where computational analysis and human contextual judgment can play complementary roles. Its distinguishing ideas are:
- **Human contextual understanding:** Operators may contribute empathy, cultural sensitivity, and local knowledge.
- **Explainable recommendations:** Reasoning and alternatives are made visible for review; an explanation supports scrutiny but does not guarantee correctness.
- **Trust-aware interaction:** A simple trust score affects simulated decision rules and is updated after outcomes.
- **Human judgment in exceptional cases:** The model assigns human leadership to exceptional and ethical decisions.

## Potential advantages and limitations
**Potential advantages**
- Combines structured computational support with human contextual judgment.
- Makes a distinction between routine decisions and situations that call for human review.
- Records explanations, alternatives, and trust updates for inspection.
- Provides a framework for discussing accountability and ethical considerations.

**Limitations and risks**
- Skilled human operators require training and may not be available at all times.
- Human involvement can increase decision time and limit scale.
- Human judgment can introduce bias, and AI recommendations can also encode bias.
- Trust scores and performance scores in this notebook are heuristic; real deployment would require careful definition, measurement, and validation.
- The simulation does not test whether collaboration improves outcomes relative to human-only or AI-only decision-making.
- Integrating operator feedback into a production system requires governance, privacy protection, and monitoring beyond the scope of this example.

## When this approach may be useful
A human-AI partnership may be worth considering when customer-facing decisions involve meaningful contextual uncertainty, exceptional circumstances, ethical trade-offs, or high service expectations. Any real implementation should define decision authority, escalation procedures, data protections, evaluation measures, and mechanisms for auditing outcomes.